### What needs to be done for the gold layer (Phase 1)
- transfer the control exception table to the gold layer. 
- transfer control synthetic table to the gold layer.
- transfer data dictionary table to the gold layer.
- transfer control logic table to the gold layer.

import the libraries

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import TimestampType
import requests
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType, FloatType, DateType, TimestampType

Read the required

In [0]:
%sql

--DROP TABLE IF EXISTS curate_control_datalake.dev.curate_control_logic;

In [0]:
control_logic = spark.sql("select * from process_control_datalake.dev.processed_control_logic")
control_exception = spark.read.table("process_control_datalake.dev.processed_control_exception")
control_data_dictionary = spark.sql("select * from process_control_datalake.dev.processed_data_dictionary")
control_synthetic_data = spark.read.table("process_control_datalake.dev.processed_synthetic_data")

In [0]:
control_logic.write.mode("overwrite").saveAsTable("curate_control_datalake.dev.curate_control_logic")

Create Gold layer tables 

Create the dev database

In [0]:
catalog = "curate_control_datalake"
database = "dev"
spark.sql(f"create database if not exists {catalog}.{database}")
spark.sql(f"use {catalog}.{database}")

write the table

In [0]:
control_logic.write.mode("overwrite").saveAsTable("curate_control_datalake.dev.curate_control_logic")
control_exception.write.mode("overwrite").saveAsTable("curate_control_datalake.dev.curate_control_exception")
control_data_dictionary.write.mode("overwrite").saveAsTable("curate_control_datalake.dev.curate_data_dictionary")
control_synthetic_data.write.mode("overwrite").saveAsTable("curate_control_datalake.dev.curate_synthetic_data")


### What needs to be done for the gold layer (Phase 2)
- Push the information to a Postgres database(SUPABASE) using a Fast API

setting up the url

In [0]:
table = ""
SUPABASE="https://controlweb-supabase.azurewebsites.net/insert/{table}"




reading the tables

In [0]:
synthetic_data = spark.read.table("curate_control_datalake.dev.curate_synthetic_data")
data_dictionary = spark.read.table("curate_control_datalake.dev.curate_data_dictionary")
control_logic = spark.read.table("curate_control_datalake.dev.curate_control_logic")
control_exception = spark.read.table("curate_control_datalake.dev.curate_control_exception")

convert the tables to list of dictionary

In [0]:
control_logic.collect()[0].asDict()

In [0]:
control_logic.columns

In [0]:


control_logic.printSchema()

A function for sending the data to an external source

In [0]:
synthetic_data.printSchema()

In [0]:
check = 1
print(SUPABASE.format(table=check))

In [0]:
def send_data(table, table_name:str):
    payload = [row.asDict() for row in table.collect()]
    target_destination = SUPABASE.format(table=table_name)
    response = requests.post(target_destination, json=payload)
    return response.json() 



In [0]:
print(send_data(data_dictionary, "control_dictionary"))
print(send_data(synthetic_data, "synthetic_data"))
print(send_data(control_logic, "control_logic"))
print(send_data(control_exception, "control_exception"))